# DermaScreen: train the skin photo model (MobileNetV2)

Run the cells **top to bottom** in Google Colab. Menu: *Runtime > Change runtime type > T4 GPU* first.

What this notebook does
1. Downloads two licensed research datasets (**Fitzpatrick17k** and **Google SCIN**), not random Google images.
2. **Counts** the images per condition, so you only train the conditions that have enough data.
3. Trains a MobileNetV2 model (transfer learning), splitting by case so test images are never seen in training.
4. Measures honest accuracy: per-condition recall, confusion matrix, top-3 accuracy.
5. Exports `dermascreen.tflite` and `labels.json` for your backend's `models/` folder.

Rules for the images: they stay inside Colab. Do **not** upload them to GitHub, put them in your app, or show them to users.
Fitzpatrick17k is CC BY-NC-SA 3.0 (non-commercial, give credit). SCIN is under the SCIN Data Use License (never try to identify contributors).
Cite both datasets in your capstone paper.

In [ ]:
import os, re, io, ast, json, random, subprocess, concurrent.futures as cf
import numpy as np, pandas as pd, requests
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Everything is saved in your Google Drive, so it survives if Colab disconnects.
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/dermascreen/data'
os.makedirs(WORK, exist_ok=True)

# ---- settings you may change ----
MIN_IMAGES    = 80     # a condition with fewer usable images than this is dropped (kept as information only)
MAX_PER_CLASS = 500    # use at most this many images per condition (keeps classes balanced)
OTHER_MAX     = 600    # images of other conditions, used for the 'other' class
IMG           = 224

# Our condition codes (they MUST match skin_conditions.code in your database) and the
# words that identify them in each dataset's label text. Edit these after you look at the label lists below.
PATTERNS = {
    'acne':             r'^acne',
    'eczema':           r'^eczema|atopic dermatitis',
    'tinea':            r'^tinea$|^tinea (corporis|pedis|cruris|manus|faciei)|ringworm',
    'scabies':          r'scabies',
    'impetigo':         r'impetigo',
    'melasma':          r'melasma',
    'psoriasis':        r'^psoriasis',
    'urticaria':        r'urticaria|hives',
    'vitiligo':         r'vitiligo',
    'folliculitis':     r'folliculitis',
    'molluscum':        r'molluscum',
    'warts':            r'wart|verruca',
    'tinea_versicolor': r'versicolor',
}
EXCLUDE = {'acne': r'keloidalis|excori|rosacea', 'tinea': r'versicolor'}

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def map_label(label):
    l = str(label).lower().strip()
    for code, pat in PATTERNS.items():
        if re.search(pat, l) and not (code in EXCLUDE and re.search(EXCLUDE[code], l)):
            return code
    return None

def save_small(img_bytes, path, max_side=320):
    im = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    im.thumbnail((max_side, max_side))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    im.save(path, 'JPEG', quality=90)

rows = []   # every image we manage to download: path, code, source, group
print('ready')

## Dataset 1: Fitzpatrick17k
The CSV lists 114 conditions with a link to each image (some links are broken, which is normal).

In [ ]:
FZ_CSV = 'https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv'
fz = pd.read_csv(FZ_CSV)
print(fz.shape); print(list(fz.columns))
print(fz['label'].value_counts().head(70).to_string())

In [ ]:
fz['code'] = fz['label'].map(map_label)
print(fz['code'].value_counts(dropna=False).to_string())
# If a condition you expect shows 0 or very few, find its real label name in the list above and fix PATTERNS.

In [ ]:
def pick(df, code, n):
    d = df[df['code'] == code]
    return d.sample(min(n, len(d)), random_state=SEED)

fz_sel = pd.concat([pick(fz, c, MAX_PER_CLASS) for c in PATTERNS])
other = fz[fz['code'].isna()].sample(OTHER_MAX // 2, random_state=SEED).assign(code='other')
fz_sel = pd.concat([fz_sel, other])
print(len(fz_sel), 'images to try')

def get_fz(r):
    path = f"{WORK}/fitz/{r['code']}/{r['md5hash']}.jpg"
    if os.path.exists(path):
        return {'path': path, 'code': r['code'], 'source': 'fitz', 'group': 'fz_' + r['md5hash']}
    try:
        resp = requests.get(r['url'], timeout=15, headers={'User-Agent': 'Mozilla/5.0 (research download)'})
        if resp.status_code != 200 or len(resp.content) < 2000:
            return None
        save_small(resp.content, path)
        return {'path': path, 'code': r['code'], 'source': 'fitz', 'group': 'fz_' + r['md5hash']}
    except Exception:
        return None

with cf.ThreadPoolExecutor(16) as ex:
    got = [g for g in ex.map(get_fz, [r for _, r in fz_sel.iterrows()]) if g]
rows += got
print('downloaded', len(got), 'of', len(fz_sel))

## Dataset 2: Google SCIN
Photos taken by ordinary people with phones, which looks much more like your app's users. If a path below fails, run the first cell to see what is in the bucket and adjust the names.

In [ ]:
BUCKET = 'dx-scin-public-data'
print(sh(f'gsutil ls gs://{BUCKET}/').stdout)
print(sh(f'gsutil ls gs://{BUCKET}/dataset/ | head -20').stdout)

In [ ]:
os.makedirs(f'{WORK}/scin', exist_ok=True)
for f in ['scin_cases.csv', 'scin_labels.csv']:
    r = sh(f'gsutil cp gs://{BUCKET}/dataset/{f} {WORK}/scin/{f}')
    print(f, r.returncode, r.stderr[:200])
cases  = pd.read_csv(f'{WORK}/scin/scin_cases.csv')
labels = pd.read_csv(f'{WORK}/scin/scin_labels.csv')
print(cases.shape, labels.shape)
print([c for c in cases.columns if 'image' in c.lower()][:6])
print([c for c in labels.columns if 'condition' in c.lower()])

In [ ]:
sc = cases.merge(labels, on='case_id')

def top_label(v):
    # weighted_skin_condition_label looks like "{'Eczema': 0.6, 'Allergic contact dermatitis': 0.4}"
    try:
        d = ast.literal_eval(v) if isinstance(v, str) else {}
        if not d: return None, 0.0
        k = max(d, key=d.get)
        return k, float(d[k])
    except Exception:
        return None, 0.0

col = 'weighted_skin_condition_label'
tl = sc[col].map(top_label)
sc['top_label'] = [t[0] for t in tl]
sc['top_w']     = [t[1] for t in tl]
sc['code'] = sc['top_label'].map(map_label)
sc_ok = sc[(sc['top_w'] >= 0.5)]          # only cases where the dermatologists agree enough
print(sc_ok['code'].value_counts(dropna=False).head(30).to_string())

In [ ]:
from google.cloud import storage
client = storage.Client.create_anonymous_client()
bucket = client.bucket(BUCKET)

img_col = 'image_1_path'
sc_sel = pd.concat([pick(sc_ok, c, MAX_PER_CLASS) for c in PATTERNS] +
                   [sc_ok[sc_ok['code'].isna()].sample(OTHER_MAX // 2, random_state=SEED).assign(code='other')])
print(len(sc_sel), 'images to try')

def get_sc(r):
    path = f"{WORK}/scin_img/{r['code']}/{r['case_id']}.jpg"
    if os.path.exists(path):
        return {'path': path, 'code': r['code'], 'source': 'scin', 'group': f"sc_{r['case_id']}"}
    try:
        data = bucket.blob(str(r[img_col])).download_as_bytes()
        save_small(data, path)
        return {'path': path, 'code': r['code'], 'source': 'scin', 'group': f"sc_{r['case_id']}"}
    except Exception:
        return None

with cf.ThreadPoolExecutor(16) as ex:
    got = [g for g in ex.map(get_sc, [r for _, r in sc_sel.iterrows()]) if g]
rows += got
print('downloaded', len(got), 'of', len(sc_sel))

## Count what we really have, then decide which conditions to train
This table is the most important result so far. Conditions below `MIN_IMAGES` are dropped from the model (they stay in the app as information).

In [ ]:
df = pd.DataFrame(rows).drop_duplicates('path')
counts = df.groupby('code').size().sort_values(ascending=False)
print(counts.to_string())

keep = [c for c, n in counts.items() if n >= MIN_IMAGES]
dropped = [c for c in counts.index if c not in keep]
print('\nTRAINING THESE:', keep)
print('TOO FEW IMAGES (info only):', dropped)
if 'other' not in keep:
    raise SystemExit("The 'other' class has too few images, check the downloads above.")

df = df[df['code'].isin(keep)].copy()
# balance: no class larger than MAX_PER_CLASS
df = df.groupby('code', group_keys=False).apply(lambda g: g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED))
class_names = sorted(keep)
class_names.remove('other'); class_names.append('other')     # 'other' always last
idx = {c: i for i, c in enumerate(class_names)}
df['y'] = df['code'].map(idx)
print(class_names)

## Split by case (no image of the same case in two sets)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
df = df.reset_index(drop=True)
fold = np.zeros(len(df), dtype=int)
for k, (_, te) in enumerate(sgkf.split(df, df['y'], df['group'])):
    fold[te] = k
df['split'] = np.where(fold == 0, 'test', np.where(fold == 1, 'val', 'train'))
print(df.groupby(['split', 'code']).size().unstack(0).to_string())

## Build the input pipeline and the model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
tf.random.set_seed(SEED)
print('GPU:', tf.config.list_physical_devices('GPU'))

BATCH = 32
augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
])

def load(path, y):
    im = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    im = tf.image.resize(im, (IMG, IMG))
    return im, y

def make_ds(part, train):
    d = df[df['split'] == part]
    ds = tf.data.Dataset.from_tensor_slices((d['path'].values, d['y'].values.astype('int32')))
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        ds = ds.shuffle(2000, seed=SEED)
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    # MobileNetV2 expects pixels scaled to [-1, 1]. Your backend does the same (x/127.5 - 1).
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds('train', True), make_ds('val', False), make_ds('test', False)

base = MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False
inp = layers.Input((IMG, IMG, 3))
x = base(inp, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(len(class_names), activation='softmax')(x)
model = models.Model(inp, out)

from sklearn.utils.class_weight import compute_class_weight
ytr = df[df['split'] == 'train']['y'].values
cw = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=ytr)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print(class_weight)

In [ ]:
# Stage 1: train only the new top layer
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy')]
h1 = model.fit(train_ds, validation_data=val_ds, epochs=12, class_weight=class_weight, callbacks=cb)

In [ ]:
# Stage 2: fine-tune the top of MobileNetV2 slowly (batch-norm layers stay frozen)
base.trainable = True
for l in base.layers[:-40]:
    l.trainable = False
for l in base.layers:
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=12, class_weight=class_weight, callbacks=cb)

## Honest evaluation on images the model has never seen

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

probs = model.predict(test_ds)
y_true = df[df['split'] == 'test']['y'].values
y_pred = probs.argmax(1)

acc  = float((y_true == y_pred).mean())
top3 = float(np.mean([t in np.argsort(p)[-3:] for t, p in zip(y_true, probs)]))
print(f'Test accuracy: {acc:.3f}   Top-3 accuracy: {top3:.3f}')
rep = classification_report(y_true, y_pred, labels=range(len(class_names)), target_names=class_names, output_dict=True, zero_division=0)
print(classification_report(y_true, y_pred, labels=range(len(class_names)), target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=90)
ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black', fontsize=8)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion matrix (test set)')
plt.tight_layout(); plt.savefig('/content/drive/MyDrive/dermascreen/export/confusion_matrix.png', dpi=150); plt.show()
# Read it: which conditions get mixed up (tinea vs eczema vs psoriasis is normal)? Put this figure in your paper.

## Confidence check
The app shows **No clear match** when confidence is under 0.40. This shows how many test images are above that line and how often they are right.

In [ ]:
conf = probs.max(1)
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    m = conf >= t
    print(f'threshold {t}: answers {m.mean()*100:5.1f}% of images, correct {(y_true[m]==y_pred[m]).mean()*100 if m.any() else 0:5.1f}% of those')

## Export for the backend
The files are saved in **Google Drive > My Drive > dermascreen > export**, so you can also download them from there at any time.


In [ ]:
os.makedirs('/content/drive/MyDrive/dermascreen/export', exist_ok=True)
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tfl = conv.convert()
open('/content/drive/MyDrive/dermascreen/export/dermascreen.tflite', 'wb').write(tfl)
json.dump(class_names, open('/content/drive/MyDrive/dermascreen/export/labels.json', 'w'))
json.dump({'test_accuracy': acc, 'top3_accuracy': top3, 'classes': class_names,
           'per_class_recall': {c: rep[c]['recall'] for c in class_names},
           'images': df.groupby('code').size().to_dict(),
           'sources': ['Fitzpatrick17k (CC BY-NC-SA 3.0)', 'Google SCIN (SCIN Data Use License)']},
          open('/content/drive/MyDrive/dermascreen/export/metrics.json', 'w'), indent=2)

# sanity check: the .tflite file must agree with the Keras model
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
d_in, d_out = it.get_input_details()[0], it.get_output_details()[0]
xb, _ = next(iter(test_ds))
it.set_tensor(d_in['index'], xb[:1].numpy()); it.invoke()
print('keras :', np.round(model.predict(xb[:1], verbose=0)[0], 3))
print('tflite:', np.round(it.get_tensor(d_out['index'])[0], 3))
print('size  :', round(len(tfl) / 1e6, 1), 'MB')

print('''
-- Run this in pgAdmin (as postgres) to record the new model:
UPDATE skin_classification_schema.model_versions SET is_active = FALSE WHERE is_active;
INSERT INTO skin_classification_schema.model_versions (name, version, training_data, accuracy, is_active, trained_at)
VALUES ('MobileNetV2', '1.0', 'Fitzpatrick17k + Google SCIN', %.4f, TRUE, CURRENT_DATE);
''' % acc)

In [ ]:
from google.colab import files
for f in ['dermascreen.tflite', 'labels.json', 'metrics.json']:
    files.download('/content/drive/MyDrive/dermascreen/export/' + f)
files.download('/content/drive/MyDrive/dermascreen/export/confusion_matrix.png')

## After the download
1. Put `dermascreen.tflite` and `labels.json` in your backend folder: `backend/models/`.
2. Run the SQL printed above in pgAdmin.
3. In the backend terminal: `python -m pip install tflite-runtime` (if that fails: `python -m pip install tensorflow`), then restart uvicorn.
4. Submit a screening: the result page now says the photo model was used.
5. Keep `metrics.json` and the confusion matrix for your paper.